# M1 改进实验 · YOLOv11n + EMA 注意力（Colab）

在基线主干末端插入 EMA 多尺度注意力，针对 standing/sitting、walk/active 姿态混淆。

操作同基线笔记本：选 T4 GPU → 全部运行 → 约 2 小时 → 下载 results.zip。
第 4 格会先做**结构自检**（几秒），通过后才进入训练，发现配置错误不浪费 GPU 时间。

In [ ]:
!nvidia-smi

In [ ]:
# 下载数据集（与基线相同）
API_KEY = 'YOUR_ROBOFLOW_API_KEY'

import json, glob, zipfile, urllib.request, subprocess

meta = json.load(urllib.request.urlopen(
    f'https://api.roboflow.com/km-sd0ce/pig-behavior-wlvku/1/yolov8?api_key={API_KEY}'))
subprocess.run(['curl', '-sL', '-o', '/content/dataset.zip', meta['export']['link']], check=True)
with zipfile.ZipFile('/content/dataset.zip') as z:
    z.extractall('/content/dataset')
DATA_YAML = glob.glob('/content/dataset/**/data.yaml', recursive=True)[0]
print('data.yaml:', DATA_YAML)

In [ ]:
!pip install -q ultralytics
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
# 定义 EMA 模块 → 注册进 ultralytics → 写入改进 yaml → 结构自检
import torch
from torch import nn
import ultralytics.nn.tasks as tasks

class EMA(nn.Module):
    """Efficient Multi-Scale Attention (ICASSP 2023)，输入输出通道数不变。"""
    def __init__(self, channels, factor=8):
        super().__init__()
        self.groups = factor
        assert channels // self.groups > 0
        self.softmax = nn.Softmax(-1)
        self.agp = nn.AdaptiveAvgPool2d((1, 1))
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.gn = nn.GroupNorm(channels // self.groups, channels // self.groups)
        self.conv1x1 = nn.Conv2d(channels // self.groups, channels // self.groups, 1)
        self.conv3x3 = nn.Conv2d(channels // self.groups, channels // self.groups, 3, padding=1)

    def forward(self, x):
        b, c, h, w = x.size()
        group_x = x.reshape(b * self.groups, -1, h, w)
        x_h = self.pool_h(group_x)
        x_w = self.pool_w(group_x).permute(0, 1, 3, 2)
        hw = self.conv1x1(torch.cat([x_h, x_w], dim=2))
        x_h, x_w = torch.split(hw, [h, w], dim=2)
        x1 = self.gn(group_x * x_h.sigmoid() * x_w.permute(0, 1, 3, 2).sigmoid())
        x2 = self.conv3x3(group_x)
        x11 = self.softmax(self.agp(x1).reshape(b * self.groups, -1, 1).permute(0, 2, 1))
        x12 = x2.reshape(b * self.groups, c // self.groups, -1)
        x21 = self.softmax(self.agp(x2).reshape(b * self.groups, -1, 1).permute(0, 2, 1))
        x22 = x1.reshape(b * self.groups, c // self.groups, -1)
        weights = (torch.matmul(x11, x12) + torch.matmul(x21, x22)).reshape(b * self.groups, 1, h, w)
        return (group_x * weights.sigmoid()).reshape(b, c, h, w)

tasks.EMA = EMA  # 注册，yaml 解析器才能找到

YAML_TEXT = r'''
nc: 10
scales:
  n: [0.50, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]
  - [-1, 1, EMA, [256]]
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 14], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 11], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]
  - [[17, 20, 23], 1, Detect, [nc]]
'''
with open('/content/yolo11-ema-n.yaml', 'w') as f:
    f.write(YAML_TEXT.strip() + '\n')

from ultralytics import YOLO
model = YOLO('/content/yolo11-ema-n.yaml')
model.load('yolo11n.pt')  # 迁移主干预训练权重，不匹配层自动跳过
with torch.no_grad():
    _ = model.model.eval()(torch.zeros(1, 3, 640, 640))
print('结构自检通过')

In [ ]:
# 训练 M1（与基线完全同参：100 轮 / 640 / batch16，保证公平对比）
model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16,
            device=0, project='/content/results', name='m1-ema')

In [ ]:
# 评估 + 保存指标
import json

metrics = model.val()
summary = {
    'mAP50': round(float(metrics.box.map50), 4),
    'mAP50-95': round(float(metrics.box.map), 4),
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
}
with open('/content/results/m1-ema/metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(summary)
print('对照基线: mAP50=0.5706  mAP50-95=0.4169  P=0.5132  R=0.5868')

In [ ]:
# 打包下载
import shutil
from google.colab import files

shutil.make_archive('/content/results', 'zip', '/content/results')
files.download('/content/results.zip')